# nbskill

> Read, edit, execute, and review nbdev notebook source from Python agents.

#| export
`nbskill` gives Python agents and maintainers tools to read, edit, run, and review nbdev notebooks by chapter, symbol, or cell ID.

In [1]:
#| exportd
from pyskills import doc, list_pyskills
from nbskill import skill
from nbskill.foundation import cell_source, parse_cells
from nbskill.graph import notebook_knowledge_graph_data
from nbskill.read import context

#| export
## Why it exists

#| export
`nbskill` validates notebook edits before writing and keeps existing cell IDs stable. Use focused execution and source diffs to check what changed.

## Learn the problem areas

Start with the [Pyskill workflow](14_pyskill.ipynb). The source notebooks contain the implementation, examples, and tests for [reading](01_read.ipynb), [editing](02_edit.ipynb), [execution](03_execute.ipynb), and [review](04_review.ipynb). The [editing notebook](02_edit.ipynb) documents validation, export, and rollback for batch edits.

## Choose a tool

| Need | Public operation |
| --- | --- |
| Find the relevant chapter or symbol | `context` |
| Route generated Python back to its source notebook | `generated_owner` |
| Make an atomic edit using stable cell IDs | `edit_notebook` |
| Run focused checks in the project environment | `exec_nb` |
| Review changed cell source and notebook style | `diff_nb`, `style_check` |
| Assess execution evidence after a change | `notebook_state` |

Load `nbskill.skill` through Pyskills, then read `doc()` for the callable you intend to use. Its signature and parameter documentation are the operation's contract.

## Parse notebook cells

`parse_cells` accepts a compact text format with `%%markdown`, `%%code`, and `---` boundaries. This is useful when proposing several cells without notebook JSON. The result retains the two different cell types:

In [2]:
cell_text = "\n".join(["%%markdown", "## Demo", "---", "%%code", "value = 42"])
parsed_cells = parse_cells(cell_text)
assert [cell.cell_type for cell in parsed_cells] == ["markdown", "code"]
[(cell.cell_type, cell_source(cell)) for cell in parsed_cells]

[('markdown', '## Demo'), ('code', 'value = 42')]

## Read a notebook

Use `context` to get stable cell addresses. Start with a summary, then select a chapter or symbol for the full source.

In [ ]:
#| hide_output
notebook_context = context("nbs/14_pyskill.ipynb", view="summary", verbose=False)
assert notebook_context["selection"]["matches"]
notebook_context

#| export
## Load the Pyskill

#| export
Install `nbskill==0.0.21` from its GitHub release tag in the Python environment used by the agent. With uv, pin both the package version and its release source:

```toml
[project]
dependencies = ["nbskill==0.0.21"]

[tool.uv.sources]
nbskill = {git = "https://github.com/MorsCerta-crypto/nbskill.git", tag = "0.0.21"}
```

Authenticate private-repository access with `gh auth login` and `gh auth setup-git`, commit `uv.lock`, and install with `uv sync --locked`. For development inside this repository, use `uv sync --locked` and the checks in [DEVELOPMENT.md](DEVELOPMENT.md). The package registers `nbskill.skill` in the `pyskills` entry-point group. Discovery lists its scope without importing every problem area. Inspect the module and then the selected operation:

In [ ]:
#| hide_output
#| exportd
assert "nbskill.skill" in list_pyskills()
doc(skill.context)

## Use with aai-coding

`aai-coding` owns the persistent session and ordinary Python files. `nbskill.skill` owns notebook source, generated modules, focused execution, and notebook review. `generated_owner` chooses the route for each Python file; a repository marker alone does not make every file generated.

`install_nbskill --target codex` installs the local notebook routing skill. The Python Pyskill works independently of that host-specific installation.

## Trace notebook relationships

The local graph links notebooks, cells, headings, symbols, and imports. Definition and call evidence helps a maintainer find related source before changing a symbol. Similarity edges are hints rather than runtime dependency guarantees. This example displays a small summary of the real source tree. Run it manually because building the graph scans all source notebooks:

In [ ]:
#| eval: false
graph = notebook_knowledge_graph_data("nbs")
dict(nodes=len(graph["nodes"]), edges=len(graph["edges"]), node_types=sorted({node["type"] for node in graph["nodes"]}), edge_types=sorted({edge["type"] for edge in graph["edges"]}))

## Development

Use the [development guide](DEVELOPMENT.md) for checks and releases, and [WORKFLOW.md](WORKFLOW.md) for the shared branch and review policy.